# Uzbek translation audio — transcription on Colab

Stage 1 of `scripts/uz_audio/` (see its README and `docs/MULTILINGUAL_PLAN.md` §8.1),
run on a free Colab GPU instead of the Mac.

**What it does:** it downloads the 114 surah mp3s from newbooks.uz to *Colab's own temporary disk*,
transcribes each one with Whisper large-v3-turbo (word timestamps), and measures its pause profile.
It writes exactly the files the local tool expects:

| File | Contents |
|---|---|
| `transcripts/NNN.json` | segments + words, times in seconds |
| `energy/NNN.npy` | RMS dB per 20 ms hop, for the pause detector |

**Resumable:** results go to Google Drive (`MyDrive/quranayat_uz_audio/`) after every surah. If Colab
disconnects, run all cells again and finished surahs are skipped. The audio itself never goes to
Drive or anywhere else, and it disappears with the Colab session.

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Then *Runtime → Run all*.
Expect roughly 3–5 hours for all 33.8 hours of audio on a T4 with large-v3 (1.5–3 h with turbo).
A free session may disconnect before that; just reopen and *Run all*, and it continues.

**Afterwards, on the Mac** (seconds, low memory):
```bash
unzip -o ~/Downloads/uz_audio_colab.zip -d ~/.cache/quranayat/uz_audio
python3 -m scripts.uz_audio.align && python3 -m scripts.uz_audio.export
```

## 1. GPU check

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert gpu.returncode == 0, "No GPU: Runtime -> Change runtime type -> T4 GPU, then run again."
print(gpu.stdout.strip())

## 2. Install

In [ ]:
!pip -q install "faster-whisper>=1.1.1,<2"

## 3. Where results go (Google Drive, so a disconnect loses nothing)

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/quranayat_uz_audio"
MP3 = "/content/mp3"  # Colab's temporary disk: never Drive
for d in (f"{OUT}/transcripts", f"{OUT}/energy", MP3):
    os.makedirs(d, exist_ok=True)
done = sorted(f for f in os.listdir(f"{OUT}/transcripts") if f.endswith(".json"))
print(f"{len(done)} of 114 surahs already transcribed")

## 4. Pick the model (about 5 min, optional but recommended)

There is no "Whisper v4": OpenAI's newest are **large-v3** and **large-v3-turbo** (large-v3 with 4
decoder layers instead of 32, which makes it faster but usually a bit less accurate on Uzbek). There is
also a community **Uzbek fine-tune** of turbo. For our cuts, what matters is *where segments start*,
more than spelling.

This cell fetches only the 104 s of al-Baqarah around the three cuts approved by ear (2:284 starts at
9034.1 s, 2:285 at 9067.4 s, 2:286 at 9114.2 s), using one ~1.2 MB Range request. It runs each
candidate and prints its text plus how far its nearest segment start lands from each true cut.
Set `MODEL` in the next cell to the winner: closest starts first, readable text second.

In [ ]:
import requests, time, numpy as np
from faster_whisper import WhisperModel, decode_audio

A, B = 9018, 9122  # seconds; al-Baqarah is 96 kbps -> 12 000 bytes/s, 432-byte frames
lo, hi = A * 12000 // 432 * 432, B * 12000 // 432 * 432
r = requests.get("https://newbooks.uz/audio/t1/002.mp3", timeout=120,
                 headers={"User-Agent": "Mozilla/5.0", "Range": f"bytes={lo}-{hi - 1}"})
open("/content/window.mp3", "wb").write(r.content)
offset = lo / 12000
window = decode_audio("/content/window.mp3", sampling_rate=16000)
TRUE_CUTS = [9034.1, 9067.4, 9114.2]

CANDIDATES = ["large-v3", "large-v3-turbo", "hostmepanda/whisper-large-v3-turbo-uzbek-ct2"]
for name in CANDIDATES:
    try:
        m = WhisperModel(name, device="cuda", compute_type="float16")
    except Exception as err:
        print(f"{name}: could not load ({err})"); continue
    t0 = time.time()
    segs = list(m.transcribe(window, language="uz", word_timestamps=True,
                             condition_on_previous_text=False, vad_filter=False)[0])
    starts = [offset + s.start for s in segs]
    errs = [min(abs(st - c) for st in starts) for c in TRUE_CUTS]
    print(f"\n=== {name}   {time.time() - t0:.0f}s   cut errors: "
          + ", ".join(f"{e:.2f}s" for e in errs) + f"   (mean {np.mean(errs):.2f}s)")
    for s in segs:
        print(f"  {offset + s.start:8.2f}  {s.text.strip()}")
    del m

## 5. Download the mp3s (resumable, checked against the server's size)

In [ ]:
import concurrent.futures as cf
import time
import requests

SOURCE = "https://newbooks.uz/audio/t1/{:03d}.mp3"
HEADERS = {"User-Agent": "Mozilla/5.0"}


def fetch(n):
    url, path = SOURCE.format(n), f"{MP3}/{n:03d}.mp3"
    want = int(requests.head(url, headers=HEADERS, timeout=60).headers["Content-Length"])
    for attempt in range(8):
        have = os.path.getsize(path) if os.path.exists(path) else 0
        if have == want:
            return n, want
        try:
            h = dict(HEADERS, Range=f"bytes={have}-") if have else HEADERS
            with requests.get(url, headers=h, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(path, "ab" if have else "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
        except requests.RequestException:
            time.sleep(5 * (attempt + 1))
    raise RuntimeError(f"surah {n}: download incomplete")


todo = [n for n in range(1, 115) if not os.path.exists(f"{OUT}/transcripts/{n:03d}.json")]
started = time.time()
total = 0
with cf.ThreadPoolExecutor(12) as pool:  # newbooks.uz gives ~85 KB/s per connection
    for n, size in pool.map(fetch, todo):
        total += size
print(f"{len(todo)} files, {total / 1e9:.2f} GB in {(time.time() - started) / 60:.1f} min")

## 6. Transcribe

In [ ]:
import json
import numpy as np
from faster_whisper import WhisperModel, decode_audio

# Set from section 4's comparison. large-v3 (full) is the default: on a GPU the extra
# accuracy is worth ~2-3x the time (roughly 3-5 h on a T4 for all 33.8 h; resumable).
MODEL = "large-v3"
HOP = 320  # 20 ms at 16 kHz, as scripts/uz_audio/transcribe.py
model = WhisperModel(MODEL, device="cuda", compute_type="float16")


def energy_db(x):
    n = len(x) // HOP
    frames = x[: n * HOP].reshape(n, HOP)
    return (20 * np.log10(np.sqrt((frames ** 2).mean(axis=1)) + 1e-9)).astype(np.float32)


def transcribe(n):
    audio = decode_audio(f"{MP3}/{n:03d}.mp3", sampling_rate=16000)
    segments, _ = model.transcribe(
        audio,
        language="uz",
        word_timestamps=True,
        condition_on_previous_text=False,  # one bad segment must not snowball
        vad_filter=False,  # pauses are our signal; keep every one
    )
    out = [
        {
            "start": round(s.start, 2),
            "end": round(s.end, 2),
            "text": s.text.strip(),
            "words": [[round(w.start, 2), round(w.end, 2), w.word.strip()] for w in (s.words or [])],
        }
        for s in segments
    ]
    np.save(f"{OUT}/energy/{n:03d}.npy", energy_db(audio))
    doc = {"surah": n, "model": "faster-whisper/" + MODEL, "duration": len(audio) / 16000,
           "segments": out}
    tmp = f"{OUT}/transcripts/{n:03d}.json.tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(doc, f, ensure_ascii=False)
    os.replace(tmp, f"{OUT}/transcripts/{n:03d}.json")  # only a finished surah counts as done
    os.remove(f"{MP3}/{n:03d}.mp3")  # free Colab's disk as we go
    return len(audio) / 16000, len(out)


audio_done, started = 0.0, time.time()
for n in todo:
    t0 = time.time()
    secs, segs = transcribe(n)
    audio_done += secs
    speed = audio_done / (time.time() - started)
    print(f"{n:03d}  {secs / 60:6.1f} min audio  {time.time() - t0:5.0f}s  {segs:4d} segments"
          f"  ({speed:.1f}x real time)", flush=True)
print("all done")

## 7. Check and download the results (a few MB)

In [ ]:
import shutil
from google.colab import files

missing = [n for n in range(1, 115)
           if not (os.path.exists(f"{OUT}/transcripts/{n:03d}.json") and os.path.exists(f"{OUT}/energy/{n:03d}.npy"))]
assert not missing, f"not finished yet: {missing} — run sections 5 and 6 again"
for n in (1, 2, 114):
    d = json.load(open(f"{OUT}/transcripts/{n:03d}.json", encoding="utf-8"))
    print(n, round(d["duration"]), "s,", len(d["segments"]), "segments:", d["segments"][0]["text"][:60])
zip_path = shutil.make_archive("/content/uz_audio_colab", "zip", OUT, ".")
print(zip_path, round(os.path.getsize(zip_path) / 1e6, 1), "MB")
files.download(zip_path)